# 🌳 Deforestation Detection — Fine-Tuning on Google Colab

This notebook fine-tunes the BigEarthNet ResNet-50 model for **binary forest/non-forest classification**.

### Steps covered:
1. Check GPU
2. Mount Google Drive
3. Install dependencies
4. Download BigEarthNet forest + non-forest patches
5. Build PyTorch dataset
6. Load pretrained model and fine-tune
7. Evaluate (Accuracy, Precision, Recall, F1)
8. Save weights to Drive
9. Instructions to use weights in your project

---
> **Before running:** Make sure runtime is set to **GPU**  
> Runtime → Change runtime type → T4 GPU

## ✅ Step 1 — Check GPU

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU')

Using device: cuda
GPU: Tesla T4
Memory: 15.6 GB


: 

## ✅ Step 2 — Mount Google Drive

In [4]:
import os

# ── Mount Google Drive (with retry handling) ─────────────────────
try:
    from google.colab import drive

    # If already mounted, unmount first to get a fresh auth
    if os.path.exists('/content/drive/MyDrive'):
        print('Drive already mounted.')
    else:
        print('Mounting Google Drive...')
        print('👉 A popup/link will appear asking you to sign in to Google.')
        print('   Allow it — then come back here.')
        drive.mount('/content/drive', force_remount=False)
        print('✅ Drive mounted successfully.')

except ValueError as e:
    print(f'❌ Mount failed: {e}')
    print()
    print('To fix this:')
    print('  1. Go to Runtime → Disconnect and delete runtime')
    print('  2. Then Runtime → Run all  (or re-run this cell)')
    print('  3. When the Google sign-in popup appears, complete it quickly')
    print('  4. If popup is blocked, click the link shown in the output instead')
    raise

DRIVE_DIR = '/content/drive/MyDrive/DeforestationProject'
DATA_DIR  = f'{DRIVE_DIR}/data'
MODEL_DIR = f'{DRIVE_DIR}/models'

os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Working folder: {DRIVE_DIR}')
print(f'Data folder:    {DATA_DIR}')
print(f'Models folder:  {MODEL_DIR}')


Mounted at /content/drive
✅ Drive mounted successfully.
Working folder: /content/drive/MyDrive/DeforestationProject
Data folder:    /content/drive/MyDrive/DeforestationProject/data
Models folder:  /content/drive/MyDrive/DeforestationProject/models


## ✅ Step 3 — Install Dependencies

In [5]:
!pip install -q torch torchvision safetensors huggingface_hub \
               datasets scikit-learn matplotlib tqdm numpy tensorboard seaborn

print('All dependencies installed.')


All dependencies installed.


In [ ]:
# Optional but recommended: authenticate to Hugging Face for higher rate limits
import os

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF authentication enabled.')
else:
    print('HF token not found. Continuing with public (unauthenticated) access.')


## ✅ Step 4 — Download BigEarthNet Forest + Non-Forest Patches

In [6]:
# ── Configuration ────────────────────────────────────────────────
# How many patches per class to use (increase for better accuracy)
# 3000 per class = ~6000 total — good balance of speed vs quality
PATCHES_PER_CLASS = 3000

FOREST_CLASSES = [
    'Broad-leaved forest',
    'Coniferous forest',
    'Mixed forest',
    'Transitional woodland, shrub',
    'Agro-forestry areas'
]

NON_FOREST_CLASSES = [
    'Arable land',
    'Pastures',
    'Urban fabric',
    'Industrial or commercial units',
    'Inland waters'
]

print(f'Will download up to {PATCHES_PER_CLASS} patches per class')
print(f'Forest classes:     {FOREST_CLASSES}')
print(f'Non-forest classes: {NON_FOREST_CLASSES}')

Will download up to 3000 patches per class
Forest classes:     ['Broad-leaved forest', 'Coniferous forest', 'Mixed forest', 'Transitional woodland, shrub', 'Agro-forestry areas']
Non-forest classes: ['Arable land', 'Pastures', 'Urban fabric', 'Industrial or commercial units', 'Inland waters']


In [8]:
from datasets import load_dataset, get_dataset_config_names
import json

# Safe fallback if auth cell above was skipped
HF_TOKEN = globals().get('HF_TOKEN', None)

# ── First check what configs are available ───────────────────────
print('Checking available dataset configs...')
try:
    configs = get_dataset_config_names('torchgeo/bigearthnet', token=HF_TOKEN)
    print(f'Available configs: {configs}')
except Exception as e:
    print(f'Could not fetch configs: {e}')
    configs = ['default']

# Use 'default' config (the only available one)
CONFIG_NAME = 'default'

DATASET_CACHE = f'{DATA_DIR}/bigearthnet_forest_cache'

print(f'\nLoading BigEarthNet from HuggingFace (config="{CONFIG_NAME}", streaming)...')
print('This may take 2-5 minutes on first run...')
print('Note: trust_remote_code is intentionally not used (unsupported by current datasets versions).')

# Keep this call free of trust_remote_code (deprecated/unsupported).
raw_dataset = load_dataset(
    'torchgeo/bigearthnet',
    name=CONFIG_NAME,
    split='train',
    streaming=True,     # Stream = no full download needed
    token=HF_TOKEN
)

# Peek at one sample to see available fields
sample_peek = next(iter(raw_dataset))
print(f'\nDataset fields: {list(sample_peek.keys())}')
print(f'Image type: {type(sample_peek.get("image", sample_peek.get("s2", "N/A")))}')

# Detect the image key and label key
IMAGE_KEY = 'image' if 'image' in sample_peek else ('s2' if 's2' in sample_peek else list(sample_peek.keys())[0])
LABEL_KEY = 'label' if 'label' in sample_peek else ('labels' if 'labels' in sample_peek else 'label')
print(f'Using image key: "{IMAGE_KEY}", label key: "{LABEL_KEY}"')
print('Dataset connected. Now filtering forest and non-forest patches...')


Checking available dataset configs...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'torchgeo/bigearthnet' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'torchgeo/bigearthnet' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Available configs: ['default']

Loading BigEarthNet from HuggingFace (config="default", streaming)...
This may take 2-5 minutes on first run...

Dataset fields: ['tif', '__key__', '__url__']
Image type: <class 'str'>
Using image key: "tif", label key: "label"
Dataset connected. Now filtering forest and non-forest patches...


In [10]:
import numpy as np
from tqdm.auto import tqdm
import time

forest_images     = []
non_forest_images = []
forest_labels     = []
non_forest_labels = []

BEN_CLASSES = [
    'Agro-forestry areas', 'Arable land', 'Beaches, dunes, sands',
    'Broad-leaved forest', 'Coastal wetlands', 'Complex cultivation patterns',
    'Coniferous forest', 'Industrial or commercial units', 'Inland waters',
    'Inland wetlands', 'Land principally occupied by agriculture, with significant areas of natural vegetation',
    'Marine waters', 'Mixed forest', 'Moors, heathland and sclerophyllous vegetation',
    'Natural grassland and sparsely vegetated areas', 'Pastures',
    'Permanent crops', 'Transitional woodland, shrub', 'Urban fabric'
]

def collect_patches(dataset, patches_per_class, forest_classes, non_forest_classes,
                    forest_images, non_forest_images, forest_labels, non_forest_labels,
                    image_key, label_key, skip=0):
    """Collect patches with automatic retry on network errors."""
    i = 0
    skipped = 0
    pbar = tqdm(total=patches_per_class * 2, desc='Collecting patches')
    pbar.update(len(forest_images) + len(non_forest_images))

    for attempt in range(5):   # up to 5 retries on network error
        try:
            for sample in dataset:
                i += 1

                # Skip already-processed samples on retry
                if skipped < skip:
                    skipped += 1
                    continue

                # ── Get labels ────────────────────────────────────────
                raw_lbl = sample.get(label_key, [])
                if isinstance(raw_lbl, (int, str)):
                    sample_labels = [raw_lbl]
                else:
                    sample_labels = list(raw_lbl)

                # Convert integer indices → class name strings
                if sample_labels and isinstance(sample_labels[0], int):
                    sample_labels = [BEN_CLASSES[idx] for idx in sample_labels if idx < len(BEN_CLASSES)]

                is_forest     = any(lbl in forest_classes     for lbl in sample_labels)
                is_non_forest = any(lbl in non_forest_classes for lbl in sample_labels)

                # ── Get image ─────────────────────────────────────────
                raw_img = sample.get(image_key)
                if raw_img is None:
                    continue

                img = np.array(raw_img, dtype=np.float32)

                if img.ndim == 3 and img.shape[-1] <= 13:
                    img = img.transpose(2, 0, 1)
                elif img.ndim == 2:
                    img = img[np.newaxis, :, :]

                if img.shape[0] < 4:
                    continue

                # ── Store ─────────────────────────────────────────────
                added = False
                if is_forest and len(forest_images) < patches_per_class:
                    forest_images.append(img)
                    forest_labels.append(0)
                    added = True
                elif is_non_forest and len(non_forest_images) < patches_per_class:
                    non_forest_images.append(img)
                    non_forest_labels.append(1)
                    added = True

                if added:
                    pbar.update(1)

                if len(forest_images) >= patches_per_class and len(non_forest_images) >= patches_per_class:
                    pbar.close()
                    return i   # Done!

                if i % 2000 == 0:
                    pbar.set_postfix({'forest': len(forest_images), 'non-forest': len(non_forest_images)})

            break  # finished iterating without error

        except Exception as e:
            print(f'\n⚠️  Network error at sample {i}: {e}')
            print(f'   Retrying (attempt {attempt+1}/5) in 5 seconds...')
            print(f'   Progress so far — Forest: {len(forest_images)}, Non-forest: {len(non_forest_images)}')
            skip = i   # skip already processed samples
            time.sleep(5)

            # Re-create the dataset stream for retry
            dataset = load_dataset(
                'torchgeo/bigearthnet',
                name=CONFIG_NAME,
                split='train',
                streaming=True,
                token=HF_TOKEN
            )
            i = 0
            skipped = 0

    pbar.close()
    return i


print(f'Collecting {PATCHES_PER_CLASS} forest and {PATCHES_PER_CLASS} non-forest patches...')
print(f'Using image_key="{IMAGE_KEY}", label_key="{LABEL_KEY}"')
print('This may take 15–30 minutes. Network errors will auto-retry.\n')

total_scanned = collect_patches(
    raw_dataset, PATCHES_PER_CLASS,
    FOREST_CLASSES, NON_FOREST_CLASSES,
    forest_images, non_forest_images,
    forest_labels, non_forest_labels,
    IMAGE_KEY, LABEL_KEY
)

print(f'\nDone. Scanned {total_scanned} total samples.')
print(f'  Forest patches:     {len(forest_images)}')
print(f'  Non-forest patches: {len(non_forest_images)}')
if forest_images:
    print(f'  Image shape:        {forest_images[0].shape}')


In [11]:
# Save the collected patches to Drive so you don't re-download next session
import numpy as np

all_images = np.array(forest_images + non_forest_images, dtype=np.float32)
all_labels = np.array(forest_labels + non_forest_labels, dtype=np.int64)

np.save(f'{DATA_DIR}/images.npy', all_images)
np.save(f'{DATA_DIR}/labels.npy', all_labels)

print(f'Saved to Drive:')
print(f'  images.npy shape: {all_images.shape}')
print(f'  labels.npy shape: {all_labels.shape}')
print(f'  Label 0 = Forest, Label 1 = Non-Forest')

: 

## ✅ Step 5 — Build PyTorch Dataset
> **Note:** If you already have `images.npy` saved in Drive from a previous session, run this cell to skip the download.

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

# ── Load from Drive (fast — skips re-downloading) ────────────────
all_images = np.load(f'{DATA_DIR}/images.npy')
all_labels = np.load(f'{DATA_DIR}/labels.npy')
print(f'Loaded from Drive: {all_images.shape}, labels: {all_labels.shape}')


class BigEarthNetBinary(Dataset):
    """Binary forest/non-forest dataset from BigEarthNet patches."""

    # Use 4 Sentinel-2 bands: B4(Red), B3(Green), B2(Blue), B8(NIR)
    # BigEarthNet S2 band order: B01,B02,B03,B04,B05,B06,B07,B08,B8A,B09,B11,B12
    # Indices:                     0    1    2    3    4    5    6    7    8    9   10   11
    BAND_INDICES = [3, 2, 1, 7]   # B4, B3, B2, B8 → RGB + NIR

    def __init__(self, images, labels, img_size=64):
        self.images   = torch.tensor(images, dtype=torch.float32)
        self.labels   = torch.tensor(labels, dtype=torch.long)
        self.img_size = img_size

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]

        # Select 4 bands (RGB + NIR)
        if img.shape[0] >= 8:
            img = img[self.BAND_INDICES]
        else:
            # Fallback: use first 4 bands
            img = img[:4]

        # Normalize to [0, 1] using Sentinel-2 max value (10000)
        img = img / 10000.0
        img = torch.clamp(img, 0.0, 1.0)

        # Resize to 64x64
        img = torch.nn.functional.interpolate(
            img.unsqueeze(0), size=(self.img_size, self.img_size), mode='bilinear', align_corners=False
        ).squeeze(0)

        return img, self.labels[idx]


# Build dataset and split 70% train / 15% val / 15% test
full_dataset = BigEarthNetBinary(all_images, all_labels)

total      = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Dataset ready:')
print(f'  Train: {train_size} samples')
print(f'  Val:   {val_size} samples')
print(f'  Test:  {test_size} samples')

## ✅ Step 6 — Load Pretrained Model and Fine-Tune

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = 'cuda' if torch.cuda.is_available() else 'cpu'

BIGEARTHNET_REPO = 'BIFOLD-BigEarthNetv2-0/resnet50-s2-v0.2.0'
NUM_CLASSES      = 2   # 0=Forest, 1=Non-Forest
INPUT_CHANNELS   = 4   # RGB + NIR

print('Downloading BigEarthNet pretrained weights from HuggingFace...')
weights_path = hf_hub_download(BIGEARTHNET_REPO, 'model.safetensors')
state        = load_file(weights_path)

# Build ResNet-50 with 10-band input (BigEarthNet format)
base_model         = resnet50(weights=None)
base_model.conv1   = nn.Conv2d(10, 64, kernel_size=7, stride=2, padding=3, bias=False)
base_model.fc      = nn.Linear(2048, 19)   # 19 BigEarthNet classes

# Load pretrained weights
mapped = {k[len('model.vision_encoder.'):]: v
          for k, v in state.items() if k.startswith('model.vision_encoder.')}
base_model.load_state_dict(mapped, strict=False)
print('Pretrained weights loaded.')

# ── Adapt model for 4-band input and binary output ───────────────
# Replace first conv: 10 bands → 4 bands (RGB + NIR)
original_conv      = base_model.conv1
new_conv           = nn.Conv2d(INPUT_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
with torch.no_grad():
    # Copy weights for bands B4,B3,B2 (indices 3,2,1 in 10-band)
    new_conv.weight[:, 0] = original_conv.weight[:, 3]   # Red
    new_conv.weight[:, 1] = original_conv.weight[:, 2]   # Green
    new_conv.weight[:, 2] = original_conv.weight[:, 1]   # Blue
    new_conv.weight[:, 3] = original_conv.weight[:, 7]   # NIR (B8)
base_model.conv1 = new_conv

# Replace final layer: 19 classes → 2 classes (Forest/Non-Forest)
base_model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(2048, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES)
)

# Freeze backbone — only train the new head first
for name, param in base_model.named_parameters():
    if 'fc' not in name and 'conv1' not in name:
        param.requires_grad = False

model = base_model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'Model ready on {device}')
print(f'Trainable params: {trainable:,} / {total_p:,}')

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# ── TensorBoard writer ────────────────────────────────────────────
TB_LOG_DIR = f'{MODEL_DIR}/tensorboard_logs'
os.makedirs(TB_LOG_DIR, exist_ok=True)
writer = SummaryWriter(log_dir=TB_LOG_DIR)
print(f'TensorBoard logs → {TB_LOG_DIR}')


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs     = model(imgs)
            loss        = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


print('Optimizer, training functions, and TensorBoard writer ready.')


In [ ]:
# ── Phase 1: Train only the classification head (5 epochs) ───────
PHASE1_EPOCHS = 5
best_val_loss = float('inf')
history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('='*60)
print('PHASE 1: Training classification head only')
print('='*60)

for epoch in range(1, PHASE1_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # ── Log to TensorBoard ────────────────────────────────────────
    writer.add_scalars('Loss',     {'train': train_loss, 'val': val_loss},  epoch)
    writer.add_scalars('Accuracy', {'train': train_acc,  'val': val_acc},   epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'],     epoch)
    writer.flush()

    print(f'Epoch [{epoch}/{PHASE1_EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{MODEL_DIR}/best_model_phase1.pth')
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')

print('\nPhase 1 complete.')


In [ ]:
# ── Phase 2: Unfreeze last 2 ResNet layers and fine-tune (10 epochs) ──
for name, param in model.named_parameters():
    if 'layer4' in name or 'layer3' in name or 'fc' in name:
        param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=1e-4, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

PHASE2_EPOCHS = 10
best_val_loss = float('inf')

print('='*60)
print('PHASE 2: Fine-tuning with unfrozen deeper layers')
print('='*60)

for epoch in range(1, PHASE2_EPOCHS + 1):
    global_epoch = PHASE1_EPOCHS + epoch   # continue epoch count from Phase 1

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # ── Log to TensorBoard (continuing from Phase 1 epoch count) ──
    writer.add_scalars('Loss',     {'train': train_loss, 'val': val_loss},  global_epoch)
    writer.add_scalars('Accuracy', {'train': train_acc,  'val': val_acc},   global_epoch)
    writer.add_scalar('Learning_Rate', optimizer.param_groups[0]['lr'],     global_epoch)
    writer.flush()

    print(f'Epoch [{epoch}/{PHASE2_EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{MODEL_DIR}/best_model_final.pth')
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')

writer.close()
print('\nPhase 2 complete. Best model saved to Drive.')
print(f'TensorBoard logs saved to: {TB_LOG_DIR}')


In [ ]:
# ── Launch TensorBoard inline ─────────────────────────────────────
# Run this cell AFTER training to visualize loss, accuracy, and LR curves

%load_ext tensorboard
%tensorboard --logdir {TB_LOG_DIR}

# You will see 3 charts:
#   • Loss       → train vs val loss across all 15 epochs
#   • Accuracy   → train vs val accuracy across all 15 epochs
#   • LR         → learning rate schedule (drops in Phase 2)


In [ ]:

# ── Training Graphs (placeholder — replace lists with real values after training) ─
import matplotlib.pyplot as plt

epochs = list(range(1, 16))

# Replace these with history['train_loss'] / history['val_loss'] after training
train_loss = [0.65, 0.52, 0.41, 0.34, 0.28, 0.25, 0.22, 0.20, 0.19, 0.18, 0.17, 0.17, 0.16, 0.16, 0.15]
val_loss   = [0.68, 0.55, 0.45, 0.37, 0.31, 0.28, 0.25, 0.23, 0.22, 0.21, 0.20, 0.20, 0.19, 0.19, 0.18]

# Replace these with history['train_acc'] / history['val_acc'] after training
train_acc  = [55, 63, 71, 77, 82, 85, 87, 88, 89, 90, 91, 91, 92, 92, 93]
val_acc    = [53, 61, 69, 75, 80, 83, 85, 86, 87, 88, 89, 89, 90, 90, 91]

# Replace with logged LR values after training
lr         = [1e-3]*5 + [1e-4]*5 + [5e-5]*5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Results — Deforestation Detection Model', fontsize=13, fontweight='bold')

axes[0].plot(epochs, train_loss, 'b-o', markersize=5, label='Train Loss')
axes[0].plot(epochs, val_loss,   'r-o', markersize=5, label='Val Loss')
axes[0].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6)
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, train_acc, 'b-o', markersize=5, label='Train Accuracy')
axes[1].plot(epochs, val_acc,   'r-o', markersize=5, label='Val Accuracy')
axes[1].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6)
axes[1].axhline(y=90, color='green', linestyle=':', alpha=0.7, label='90% target')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(50, 100); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].step(epochs, lr, 'g-', where='mid', linewidth=2)
axes[2].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6)
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# ── Detection Metrics: mAP, mAP_0.5:0.95, Precision ─────────────
# Replace the placeholder lists below with your real logged values after training
import matplotlib.pyplot as plt

epochs = list(range(1, 16))

# Replace with real mAP@0.5 values per epoch
map_50        = [0.30, 0.42, 0.52, 0.60, 0.66, 0.70, 0.73, 0.75, 0.77, 0.78, 0.79, 0.80, 0.81, 0.81, 0.82]

# Replace with real mAP@0.5:0.95 values per epoch
map_50_95     = [0.14, 0.20, 0.26, 0.31, 0.35, 0.38, 0.41, 0.43, 0.45, 0.46, 0.47, 0.48, 0.49, 0.49, 0.50]

# Replace with real Precision values per epoch
precision     = [0.55, 0.63, 0.69, 0.74, 0.78, 0.81, 0.83, 0.84, 0.85, 0.86, 0.87, 0.87, 0.88, 0.88, 0.89]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Detection Metrics — mAP & Precision', fontsize=13, fontweight='bold')

axes[0].plot(epochs, map_50, 'b-o', markersize=5, label='mAP@0.5')
axes[0].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
axes[0].axhline(y=0.75, color='green', linestyle=':', alpha=0.7, label='0.75 target')
axes[0].set_title('mAP @ 0.5')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('mAP')
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, map_50_95, 'r-o', markersize=5, label='mAP@0.5:0.95')
axes[1].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
axes[1].axhline(y=0.45, color='green', linestyle=':', alpha=0.7, label='0.45 target')
axes[1].set_title('mAP @ 0.5:0.95')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, precision, 'purple', marker='o', markersize=5, label='Precision')
axes[2].axvline(x=5.5, color='gray', linestyle='--', alpha=0.6, label='Phase 2 start')
axes[2].axhline(y=0.85, color='green', linestyle=':', alpha=0.7, label='0.85 target')
axes[2].set_title('Precision over Epochs')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Precision')
axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## ✅ Step 7 — Evaluate Model (Accuracy, Precision, Recall, F1)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load best weights
model.load_state_dict(torch.load(f'{MODEL_DIR}/best_model_final.pth', map_location=device))
model.eval()

all_preds, all_labels_list = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs    = imgs.to(device)
        outputs = model(imgs)
        preds   = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(labels.numpy())

all_preds        = np.array(all_preds)
all_labels_array = np.array(all_labels_list)

acc  = accuracy_score(all_labels_array,  all_preds)
prec = precision_score(all_labels_array, all_preds, average='binary')
rec  = recall_score(all_labels_array,    all_preds, average='binary')
f1   = f1_score(all_labels_array,        all_preds, average='binary')
cm   = confusion_matrix(all_labels_array, all_preds)

print('='*50)
print('TEST SET RESULTS')
print('='*50)
print(f'Accuracy:  {acc:.4f}  ({acc*100:.1f}%)')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {f1:.4f}')
print()
print(classification_report(all_labels_array, all_preds,
                             target_names=['Forest', 'Non-Forest']))

# Confusion matrix plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Forest', 'Non-Forest'],
            yticklabels=['Forest', 'Non-Forest'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Training curves
epochs_range = range(1, len(history['train_loss']) + 1)
axes[1].plot(epochs_range, history['train_acc'], label='Train Accuracy', color='blue')
axes[1].plot(epochs_range, history['val_acc'],   label='Val Accuracy',   color='orange')
axes[1].set_title('Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_results.png', dpi=150)
plt.show()
print('Plot saved to Drive.')

## ✅ Step 8 — Save Final Model to Drive

In [ ]:
import json
from datetime import datetime

# Save full checkpoint with metadata
checkpoint = {
    'model_state_dict':  model.state_dict(),
    'num_classes':       NUM_CLASSES,
    'input_channels':    INPUT_CHANNELS,
    'architecture':      'ResNet50-BigEarthNet-finetuned',
    'trained_on':        'BigEarthNet-S2 forest/non-forest subset',
    'accuracy':          float(acc),
    'f1_score':          float(f1),
    'date_trained':      datetime.now().isoformat()
}

FINAL_MODEL_PATH = f'{MODEL_DIR}/deforestation_model.pth'
torch.save(checkpoint, FINAL_MODEL_PATH)

# Save metrics as JSON for reference
metrics = {
    'accuracy':  float(acc),
    'precision': float(prec),
    'recall':    float(rec),
    'f1_score':  float(f1)
}
with open(f'{MODEL_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Final model saved: {FINAL_MODEL_PATH}')
print(f'File size: {os.path.getsize(FINAL_MODEL_PATH) / 1e6:.1f} MB')
print(f'Metrics saved: {MODEL_DIR}/metrics.json')
print()
print('Next step: Download deforestation_model.pth and copy it to:')
print('  backend/models/deforestation_model.pth')

In [ ]:
# Download the model file directly to your computer
from google.colab import files
files.download(FINAL_MODEL_PATH)
files.download(f'{MODEL_DIR}/training_results.png')
files.download(f'{MODEL_DIR}/metrics.json')
print('Download started. Check your browser downloads folder.')

## ✅ Step 9 — Use Your Trained Model in the Project

After downloading `deforestation_model.pth`:

1. Copy it to your project:
   ```
   backend/models/deforestation_model.pth
   ```

2. The project will automatically use it — `backend/src/ml/inference.py` loads from `backend/models/` first before falling back to HuggingFace.

3. Your model is now fine-tuned specifically for forest/non-forest detection!

---

### Summary of what was done:
- Downloaded BigEarthNet forest + non-forest patches (~6000 images)
- Loaded pretrained ResNet-50 weights from HuggingFace
- Adapted the model for 4-band input (RGB + NIR) and binary output
- Phase 1: Trained only the classification head (5 epochs)
- Phase 2: Fine-tuned deeper layers (10 epochs)
- Evaluated with Accuracy, Precision, Recall, F1-Score
- Saved the final model to Google Drive

## ✅ Step 10 — Run Inference on a Single Image